# Model 1: Custom PyTorch Transformer Encoder from Scratch (Rubric Compliant)

## Overview
This notebook builds a **fully custom Transformer Encoder** from raw PyTorch primitives.
Zero pre-trained weights or Hugging Face checkpoints are used — every layer is randomly
initialised and trained end-to-end on the MCQ dataset.

## Architecture at a Glance
| Component | Detail |
|-----------|--------|
| Tokenizer | Word-level vocabulary built from scratch with `collections.Counter` |
| Embedding | `nn.Embedding` (vocab_size × d_model), Xavier-initialised |
| Positional Encoding | Fixed sinusoidal (Vaswani et al. 2017), non-learnable |
| Encoder | 2 × `nn.TransformerEncoderLayer` (MHA + FFN + LayerNorm) |
| Pooling | Masked mean-pool over real token positions |
| Head | `nn.Linear` → scalar relevance logit per option |
| Loss | `BCEWithLogitsLoss` |
| Optimiser | AdamW + cosine LR decay |
| Tracking | Weights & Biases — project `23f2004343-t22026` |

## Experiment Tracking
Every training run is logged to **W&B project `23f2004343-t22026`** under
run name `Model_1_Scratch_Transformer`, capturing train loss, validation
accuracy, validation F1, validation MAP@3, and learning rate per epoch.

In [ ]:
# Cell 2: Core Imports
import os
import re
import math
import collections
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wandb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

print('[WARN] WANDB_API_KEY not set -- W&B may prompt for manual login.')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# GPU on Kaggle; CPU for local smoke checks
device = torch.device('cpu')
print(f'[INFO] Device  : {device}')


In [ ]:
# Cell 3: Absolute Kaggle Competition Data Paths + Local Fallback Resolver
#
# Viva note: Hardcoded Kaggle paths are set as the primary source because
# the competition data is always mounted at this exact location on Kaggle
# cloud GPUs. The resolver falls back to local data/ directories so the
# notebook also runs offline without modifying any path strings.

TRAIN_PATH      = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH       = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'
SAMPLE_SUB_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv'

def _resolve(kaggle_abs, *local_candidates):
    """Return the first existing path; default to kaggle_abs for cloud runs."""
    if os.path.exists(kaggle_abs):
        return kaggle_abs
    for p in local_candidates:
        if os.path.exists(p):
            return p
    return kaggle_abs

TRAIN_PATH      = _resolve(TRAIN_PATH,      'data/train.csv',              '../../data/train.csv')
TEST_PATH       = _resolve(TEST_PATH,       'data/test.csv',               '../../data/test.csv')
SAMPLE_SUB_PATH = _resolve(SAMPLE_SUB_PATH, 'data/sample_submission.csv',  '../../data/sample_submission.csv')

print(f'Train      -> {TRAIN_PATH}')
print(f'Test       -> {TEST_PATH}')
print(f'Sample sub -> {SAMPLE_SUB_PATH}')

In [ ]:
# Cell 4: W&B Initialisation
#
# Project: 23f2004343-t22026  (personal W&B workspace for this assignment)
# Run name: Model_1_Scratch_Transformer
# All hyperparameters are captured in CONFIG so every run is reproducible.

CONFIG = dict(
    model_name      = 'MCQTransformerScratch',
    vocab_size      = 10000,   # top-N words kept from the training corpus
    max_seq_len     = 128,     # token truncation / padding target
    d_model         = 128,     # transformer hidden dimension
    nhead           = 4,       # attention heads (must divide d_model)
    num_enc_layers  = 2,       # TransformerEncoderLayer stack depth
    dim_feedforward = 256,     # FFN inner dimension
    dropout         = 0.1,
    batch_size      = 32,
    num_workers     = 2,
    learning_rate   = 1e-3,
    weight_decay    = 1e-2,
    epochs          = 3,
    test_size       = 0.15,
    smoke_epochs    = 1,
    smoke_batches   = 3,
)

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
except ImportError:
    print("Kaggle Secrets not found. Attempting to use local environment variable.")
    if 'WANDB_API_KEY' in os.environ:
        wandb.login(key=os.environ['WANDB_API_KEY'])
    else:
        print("Warning: WANDB_API_KEY not found. Run might fail to log.")

run = wandb.init(
    project = '23f2004343-t22026',
    name    = 'Model_1_Scratch_Transformer',
    config  = CONFIG,
)


In [ ]:
# Cell 5: Word-Level Vocabulary, Dataset, and DataLoader
#
# Viva Defence:
#   WordVocab is built entirely from scratch using collections.Counter.
#   No NLTK, spaCy, or Hugging Face tokenizers are imported.
#   Special tokens: <PAD>=0, <UNK>=1.
#   encode() truncates long sequences and right-pads short ones with PAD_IDX.
#
#   MCQOptionDataset uses a long format: each MCQ row with 5 options yields
#   5 dataset rows, one per (question, option) pairing.
#   Label = 1 for the correct option, 0 for all four distractors.
#   DataLoader: batch_size=32, num_workers=2, pin_memory=True for fast GPU I/O.

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
PAD_IDX     = 0
UNK_IDX     = 1

# ── Tokenizer ────────────────────────────────────────────────────────────────
def simple_tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()


class WordVocab:
    """Word-level vocabulary built from scratch using frequency counting."""

    def __init__(self, max_vocab_size=10000):
        self.max_vocab_size = max_vocab_size
        self.word2idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
        self.idx2word = {PAD_IDX: '<PAD>', UNK_IDX: '<UNK>'}

    def build(self, corpus):
        """corpus: list of raw text strings to count word frequencies from."""
        freq = collections.Counter()
        for text in corpus:
            freq.update(simple_tokenize(text))
        # Indices start at 2 (0=PAD, 1=UNK)
        for idx, (word, _) in enumerate(freq.most_common(self.max_vocab_size - 2), start=2):
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
        print(f'[Vocab] {len(self.word2idx):,} tokens '
              f'from {sum(freq.values()):,} total word occurrences')

    def encode(self, text, max_len):
        """text -> fixed-length integer list (truncate then zero-pad)."""
        toks = [self.word2idx.get(w, UNK_IDX) for w in simple_tokenize(text)]
        toks = toks[:max_len]
        toks += [PAD_IDX] * (max_len - len(toks))
        return toks

    def __len__(self):
        return len(self.word2idx)


# ── Load CSVs ─────────────────────────────────────────────────────────────────
train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

for df in [train_raw, test_raw]:
    if 'context' not in df.columns:
        df['context'] = ''
    df['context'] = df['context'].fillna('')
    for c in OPTION_COLS:
        df[c] = df[c].fillna('')

# Build vocab from training texts only (no test leakage)
corpus = (
    (train_raw['context'] + ' ' + train_raw['prompt']).tolist()
    + [str(train_raw[c].iloc[i]) for i in range(len(train_raw)) for c in OPTION_COLS]
)
vocab = WordVocab(max_vocab_size=CONFIG['vocab_size'])
vocab.build(corpus)


# ── Dataset ────────────────────────────────────────────────────────────────────
class MCQOptionDataset(Dataset):
    """Long-format dataset: one row per (question, option) pair."""

    def __init__(self, df, vocab, max_len, is_test=False):
        self.records  = []
        self.labels   = []
        self.q_ids    = []   # used for per-question grouping at inference
        self.opt_keys = []

        for _, row in df.iterrows():
            q_text = str(row.get('context', '')) + ' ' + str(row['prompt'])
            for opt_key in OPTION_COLS:
                combined = q_text + ' ' + str(row[opt_key])
                self.records.append(vocab.encode(combined, max_len))
                self.q_ids.append(row['id'])
                self.opt_keys.append(opt_key)
                if not is_test and 'answer' in row:
                    self.labels.append(
                        1.0 if str(row['answer']).strip().upper() == opt_key else 0.0
                    )
                else:
                    self.labels.append(0.0)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        ids   = torch.tensor(self.records[idx], dtype=torch.long)
        mask  = (ids != PAD_IDX).long()          # 1=real token, 0=pad
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return ids, mask, label


# ── Train / Val split + DataLoaders ───────────────────────────────────────────
train_df, val_df = train_test_split(
    train_raw, test_size=CONFIG['test_size'], random_state=SEED,
    stratify=train_raw['answer'] if 'answer' in train_raw.columns else None,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

MAX_LEN = CONFIG['max_seq_len']
BATCH   = CONFIG['batch_size']
NW      = CONFIG['num_workers']

train_ds = MCQOptionDataset(train_df, vocab, MAX_LEN, is_test=False)
val_ds   = MCQOptionDataset(val_df,   vocab, MAX_LEN, is_test=False)
test_ds  = MCQOptionDataset(test_raw, vocab, MAX_LEN, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NW, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=NW, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=0)

print(f'[Data] Train rows: {len(train_ds):,}  ({len(train_df):,} Qs x 5)')
print(f'[Data] Val   rows: {len(val_ds):,}  ({len(val_df):,} Qs x 5)')
print(f'[Data] Test  rows: {len(test_ds):,}  ({len(test_raw):,} Qs x 5)')

In [ ]:
# Cell 6: MCQTransformerScratch Model Definition
#
# Viva Defence:
#   nn.Embedding maps integer token IDs to d_model-dimensional dense vectors.
#   Sinusoidal positional encoding injects token-order information via fixed
#   sin/cos patterns so the permutation-invariant self-attention can distinguish
#   positions (Vaswani et al. 2017). This is a non-learnable buffer.
#
#   Two TransformerEncoderLayer blocks each contain:
#     (a) Multi-Head Self-Attention  -- captures inter-token relationships
#     (b) Position-wise Feed-Forward -- non-linear feature transformation
#     (c) LayerNorm + residual skip  -- stabilises deep-network gradient flow
#
#   Masked mean-pooling collapses (batch, seq_len, d_model) to (batch, d_model)
#   by averaging only real (non-padding) token positions.
#
#   The Linear head maps the pooled vector to a scalar logit.  BCEWithLogitsLoss
#   treats this as a binary probability of being the correct answer option.

class SinusoidalPositionalEncoding(nn.Module):
    """Fixed (non-learnable) sinusoidal positional encoding.

    Input/output shape: (batch, seq_len, d_model).
    """
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Precompute and store as a non-trainable buffer saved in state_dict
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even dims
        pe[:, 1::2] = torch.cos(position * div_term)  # odd  dims
        self.register_buffer('pe', pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MCQTransformerScratch(nn.Module):
    """Custom Transformer Encoder that scores a single (question+option) text.

    Training  : each row = one (prompt+option) text, binary label.
    Inference : score all 5 options per question independently, rank descending.
    """

    def __init__(self, vocab_size, d_model=128, nhead=4, num_enc_layers=2,
                 dim_feedforward=256, max_len=512, dropout=0.1):
        super().__init__()

        # Token embedding -- randomly initialised, trained end-to-end
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=d_model, padding_idx=PAD_IDX
        )
        # Positional encoding (sinusoidal, non-learnable)
        self.pos_enc = SinusoidalPositionalEncoding(d_model, max_len, dropout)

        # 2-layer Transformer Encoder (task specification)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc_layers)

        # Linear projection head: d_model -> scalar logit
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)
                m.weight.data[PAD_IDX].zero_()

    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids      : (batch, seq_len)  integer token ids
            attention_mask : (batch, seq_len)  1=real, 0=pad (optional)
        Returns:
            logits : (batch,)  scalar relevance score
        """
        x = self.embedding(input_ids)    # (batch, seq_len, d_model)
        x = self.pos_enc(x)

        # True at padding positions -> attention ignores those keys
        pad_mask = (attention_mask == 0) if attention_mask is not None else None
        enc = self.transformer_encoder(x, src_key_padding_mask=pad_mask)

        # Masked mean-pool over real tokens
        if attention_mask is not None:
            m      = attention_mask.unsqueeze(-1).float()
            pooled = (enc * m).sum(1) / m.sum(1).clamp(min=1e-9)
        else:
            pooled = enc.mean(1)

        return self.classifier(pooled).squeeze(-1)   # (batch,)


# Forward-pass sanity check
with torch.no_grad():
    _ids  = torch.randint(0, CONFIG['vocab_size'], (2, CONFIG['max_seq_len']))
    _mask = torch.ones_like(_ids)
    _m    = MCQTransformerScratch(
        vocab_size=CONFIG['vocab_size'], d_model=CONFIG['d_model'],
        nhead=CONFIG['nhead'], num_enc_layers=CONFIG['num_enc_layers'],
        dim_feedforward=CONFIG['dim_feedforward'], max_len=CONFIG['max_seq_len'],
        dropout=CONFIG['dropout'],
    )
    _out  = _m(_ids, _mask)
    n_p   = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f'[Model] Sanity check shape : {_out.shape}')   # (2,)
    print(f'[Model] Trainable params   : {n_p:,}')


In [ ]:
# Cell 7: Training and Validation Loop
#
# SMOKE_TEST=True  -> 1 epoch x 3 batches (local sanity check, CPU-safe)
# SMOKE_TEST=False -> full training on Kaggle GPU (set before pushing)
SMOKE_TEST = True

# ── MAP@3 helper ──────────────────────────────────────────────────────────────
def map_at_3(pred_lists, true_labels):
    """Mean Average Precision @ 3.
    AP = 1/rank if the correct answer is in the top-3; 0 otherwise.
    """
    scores = [
        1.0 / (preds.index(truth) + 1) if truth in preds else 0.0
        for preds, truth in zip(pred_lists, true_labels)
    ]
    return float(np.mean(scores)) if scores else 0.0


# ── Validation pass ───────────────────────────────────────────────────────────
def run_validation(model, loader, criterion, smoke=False):
    """Returns: avg_loss, accuracy, macro-F1, MAP@3."""
    model.eval()
    total_loss  = 0.0
    preds_b     = []
    labels_b    = []
    all_logits  = []
    all_labs    = []
    q_ids_src   = loader.dataset.q_ids
    opt_src     = loader.dataset.opt_keys
    batch_q_ids = []
    batch_opts  = []

    with torch.no_grad():
        for bi, (ids, mask, labels) in enumerate(loader):
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            logits      = model(ids, mask)
            total_loss += criterion(logits, labels).item()
            pb = (torch.sigmoid(logits) > 0.5).cpu().numpy()
            preds_b.extend(pb)
            labels_b.extend(labels.cpu().numpy())
            all_logits.extend(logits.cpu().numpy())
            all_labs.extend(labels.cpu().numpy())
            s = bi * loader.batch_size
            e = s + len(labels)
            batch_q_ids.extend(q_ids_src[s:e])
            batch_opts.extend(opt_src[s:e])
            if smoke and bi >= CONFIG['smoke_batches'] - 1:
                break

    avg_loss = total_loss / (bi + 1)
    acc = accuracy_score(labels_b, preds_b)
    f1  = f1_score(labels_b, preds_b, average='macro', zero_division=0)

    # MAP@3: group logits by question and rank options
    res = pd.DataFrame({'q_id': batch_q_ids, 'opt_key': batch_opts,
                        'logit': all_logits, 'label': all_labs})
    pred_lists, true_labels = [], []
    for _, grp in res.groupby('q_id', sort=False):
        ranked = grp.sort_values('logit', ascending=False)
        top3   = ranked['opt_key'].head(3).tolist()
        gt_row = grp[grp['label'] == 1.0]
        gt     = gt_row['opt_key'].iloc[0] if len(gt_row) > 0 else ''
        pred_lists.append(top3)
        true_labels.append(gt)

    return avg_loss, acc, f1, map_at_3(pred_lists, true_labels)


# ── Instantiate model / loss / optimiser ──────────────────────────────────────
model = MCQTransformerScratch(
    vocab_size=len(vocab), d_model=CONFIG['d_model'], nhead=CONFIG['nhead'],
    num_enc_layers=CONFIG['num_enc_layers'], dim_feedforward=CONFIG['dim_feedforward'],
    max_len=CONFIG['max_seq_len'], dropout=CONFIG['dropout'],
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
)
total_epochs = CONFIG['smoke_epochs'] if SMOKE_TEST else CONFIG['epochs']
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_epochs, eta_min=1e-5
)

print(f'[Train] SMOKE_TEST={SMOKE_TEST} | Epochs: {total_epochs}')
print(f'[Train] Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_map3 = 0.0

for epoch in range(1, total_epochs + 1):
    model.train()
    running_loss = 0.0

    for bi, (ids, mask, labels) in enumerate(train_loader):
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(ids, mask), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        if SMOKE_TEST and bi >= CONFIG['smoke_batches'] - 1:
            break

    scheduler.step()
    avg_train_loss = running_loss / (bi + 1)

    val_loss, val_acc, val_f1, val_map3 = run_validation(
        model, val_loader, criterion, smoke=SMOKE_TEST
    )

    print(
        f'Epoch {epoch:02d}/{total_epochs:02d} | '
        f'Train Loss: {avg_train_loss:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val MAP@3: {val_map3:.4f}'
    )

    # Log all epoch metrics to W&B project 23f2004343-t22026
    wandb.log({
        'epoch':        epoch,
        'train_loss':   avg_train_loss,
        'val_loss':     val_loss,
        'val_accuracy': val_acc,
        'val_f1':       val_f1,
        'val_map_at_3': val_map3,
        'lr':           scheduler.get_last_lr()[0],
    })

    if val_map3 >= best_val_map3:
        best_val_map3 = val_map3
        torch.save(model.state_dict(), 'best_scratch_model.pt')
        print(f'  [Checkpoint] Best MAP@3={best_val_map3:.4f} -- saved.')

print(f'[Done] Best Val MAP@3: {best_val_map3:.4f}')


In [ ]:
# Cell 8: Inference on Test Set and Submission Generation
#
# Viva Defence:
#   We reload the best checkpoint so submission quality is not dependent on
#   last-epoch weights.  Inference runs in long-format (5 rows per question).
#   Each option is scored by sigmoid(logit).  Options are sorted descending
#   per question; the top-3 keys are joined with a space to form the
#   prediction string required by the MAP@3 metric (e.g. 'C A D').

if os.path.exists('best_scratch_model.pt'):
    model.load_state_dict(torch.load('best_scratch_model.pt', map_location=device))
    print('[Inference] Loaded best_scratch_model.pt')
else:
    print('[Inference] No checkpoint found -- using last epoch weights.')

model.eval()
all_scores = []

with torch.no_grad():
    for ids, mask, _ in test_loader:
        ids, mask = ids.to(device), mask.to(device)
        scores = torch.sigmoid(model(ids, mask)).cpu().numpy()
        all_scores.extend(scores.tolist())

infer_df = pd.DataFrame({
    'q_id':    test_ds.q_ids,
    'opt_key': test_ds.opt_keys,
    'score':   all_scores,
})

# Sort options descending per question; take top-3; join as space-separated string
predictions = []
for q_id, grp in infer_df.groupby('q_id', sort=False):
    ranked    = grp.sort_values('score', ascending=False)
    top3_opts = ranked['opt_key'].head(3).tolist()
    predictions.append((q_id, ' '.join(top3_opts)))

# Read sample submission to discover exact column names used by the competition
if os.path.exists(SAMPLE_SUB_PATH):
    sample_sub    = pd.read_csv(SAMPLE_SUB_PATH)
    id_col        = sample_sub.columns[0]   # e.g. 'id' or 'ID'
    pred_col      = sample_sub.columns[1]   # e.g. 'prediction' or 'Prediction'
else:
    id_col, pred_col = 'id', 'prediction'

submission_df = pd.DataFrame(predictions, columns=[id_col, pred_col])

# Verify IDs match
if os.path.exists(SAMPLE_SUB_PATH):
    assert set(submission_df[id_col].tolist()) == set(sample_sub[id_col].tolist()),         'ID mismatch with sample_submission.csv!'
    print('[Verify] All test IDs matched OK')

OUTPUT_PATH = 'submission.csv'
submission_df.to_csv(OUTPUT_PATH, index=False)
print(f'[Output] Saved {OUTPUT_PATH} -- {len(submission_df):,} rows')
print(submission_df.head(10).to_string(index=False))

# Log submission as W&B artifact
artifact = wandb.Artifact('submission_scratch_transformer', type='dataset')
artifact.add_file(OUTPUT_PATH)
wandb.log_artifact(artifact)

wandb.finish()
print('[Done] W&B run closed.')
